# V5W_02 — Graph / hypergraph build (5words) — versione snella

Riscritto da zero (le funzioni sono identiche a EEG_07f, ma senza i cell di visualizzazione a 110-parole che crashavano). Costruisce **solo `hypergraphs_pruned`** per `abs_pcc` e `plv` (tutto ciò che serve a V5W_03/04/05, incluso il cross-metrico). Solo trial `_img`. Output isolato in `data/5words_subjects/graphs/`.

**Env: `daniele_311`** · CPU (lungo: ~21k trial × 4 metriche di consensus).

Consensus filter (Iacomi): 4 metriche, soglia p95, k=2/4. Iperedge k=6.

## §1 — Config

In [ ]:
import json, logging, math, re
from pathlib import Path
import numpy as np, pandas as pd, torch
from scipy.signal import hilbert
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('v5w02')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
CSV_ROOT = project_root / 'data' / '5words_subjects'              # ISOLATO
DATA_OUT = project_root / 'data' / '5words_subjects' / 'graphs'   # ISOLATO
DATA_OUT.mkdir(parents=True, exist_ok=True)
assert CSV_ROOT.exists(), f'CSV_ROOT non trovato: {CSV_ROOT} (esegui V5W_01 + sync)'

word2label = json.loads((project_root/'configs'/'label_schemes'/'label2idx_5words.json').read_text())

# ---- CONFIG build (consensus filter Iacomi: 4 metriche, p95, k=2/4; iperedge k=6) ----
METRICS           = ['abs_pcc', 'plv']                       # output (plv -> cross-metrica V5W_05)
CONSENSUS_METRICS = ['abs_pcc', 'im_pcc', 'wpli', 'plv']
EDGE_THRESHOLD_PCT = 95
CONSENSUS_K        = 2
K_HYPEREDGE        = 6
OVERWRITE          = False
N_CHAN = 61
_PAT = re.compile(r'^P(\d+)_S(\d+)$')
log.info(f'METRICS={METRICS}  consensus={CONSENSUS_METRICS} k={CONSENSUS_K}/4 p{EDGE_THRESHOLD_PCT}  k_he={K_HYPEREDGE}')
log.info(f'Vocabolario: {word2label}')


## §2 — Metriche di connettività

In [ ]:
# Metriche di connettività (identiche a EEG_07f)
def _pcc(x):
    m = np.corrcoef(x).astype(np.float32); np.fill_diagonal(m, 0.0); return m
def _abs_pcc(x): return np.abs(_pcc(x))
def _im_pcc(x):
    z = hilbert(x, axis=1); T = z.shape[1]
    cross = (z @ z.conj().T) / T
    norms = np.sqrt(np.mean(np.abs(z)**2, axis=1)); nm = np.outer(norms, norms) + 1e-12
    m = np.imag(cross / nm).astype(np.float32); np.fill_diagonal(m, 0.0); return m
def _wpli(x):
    z = hilbert(x, axis=1); N = x.shape[0]; w = np.zeros((N, N), np.float32)
    for i in range(N):
        ci = np.imag(z[i] * np.conj(z))
        w[i] = (np.abs(np.mean(ci, axis=1)) / (np.mean(np.abs(ci), axis=1) + 1e-10)).astype(np.float32)
    np.fill_diagonal(w, 0.0); return w
def _plv(x):
    z = hilbert(x, axis=1); ph = z / (np.abs(z) + 1e-10); N = x.shape[0]; p = np.zeros((N, N), np.float32)
    for i in range(N):
        p[i] = np.abs(np.mean(ph[i] * np.conj(ph), axis=1)).astype(np.float32)
    np.fill_diagonal(p, 0.0); return p
_FNS = {'pcc': _pcc, 'abs_pcc': _abs_pcc, 'im_pcc': _im_pcc, 'wpli': _wpli, 'plv': _plv}
def compute_metric(x, metric): return _FNS[metric](x)
print('Metriche OK:', list(_FNS))


## §3 — Consensus pruning

In [ ]:
# Consensus pruning (majority vote)
def _significance_mask(mat, pct):
    N = mat.shape[0]; v = np.abs(mat); np.fill_diagonal(v, 0.0)
    thr = np.percentile(v[~np.eye(N, dtype=bool)], pct)
    mask = v >= thr; np.fill_diagonal(mask, False); return mask
def compute_consensus_mask(x, metrics, pct, k):
    eff_k = math.ceil(len(metrics)/2) if k is None else k
    N = x.shape[0]; votes = np.zeros((N, N), np.int8)
    for m in metrics: votes += _significance_mask(compute_metric(x, m), pct).astype(np.int8)
    cons = votes >= eff_k; np.fill_diagonal(cons, False); return cons
print('Consensus OK')


## §4 — Builder ipergrafo + I/O

In [ ]:
# Costruttore ipergrafo (incidenza H, top-k vicini) + I/O
def _build_incidence(adj, k):
    N = adj.shape[0]; cols = []
    for i in range(N):
        row = adj[i].copy(); row[i] = 0.0
        conn = np.where(np.abs(row) > 1e-6)[0]
        if len(conn) == 0: continue
        topk = conn[np.argsort(np.abs(row[conn]))[::-1]][:k]
        col = np.zeros(N, np.float32); col[np.concatenate([[i], topk])] = 1.0
        cols.append(col)
    return np.column_stack(cols) if cols else np.zeros((N, 0), np.float32)

def build_hypergraph_dict(x_norm, metric, k_he, label, meta, cons_mask):
    adj_full = compute_metric(x_norm, metric)
    adj_eff = adj_full * cons_mask.astype(np.float32) if cons_mask is not None else adj_full
    H = _build_incidence(adj_eff, k=k_he)
    return {'H': torch.tensor(H, dtype=torch.float32),
            'x': torch.tensor(x_norm, dtype=torch.float32),
            'adj': torch.tensor(adj_eff, dtype=torch.float32),
            'y': torch.tensor(label, dtype=torch.long), 'meta': meta}

def load_trial(p):
    x = pd.read_csv(p, header=None).values.astype(np.float32)
    if x.shape != (N_CHAN, 384): raise ValueError(f'shape {x.shape} in {p.name}')
    return x
def normalize_trial(x):
    return (x - x.mean(1, keepdims=True)) / x.std(1, keepdims=True).clip(1e-6)
def save_pt(d, p): p.parent.mkdir(parents=True, exist_ok=True); torch.save(d, p)
print('Builder + I/O OK')


## §5 — Build (driver)

In [ ]:
# DRIVER — build hypergraphs_pruned per METRICS (solo trial _img)
def out_path(metric, sid, ses, tidx):
    return DATA_OUT / f'hypergraphs_pruned_{metric}' / f'P{sid:03d}_S{ses:03d}' / f'trial_{tidx:03d}.pt'

def process_session(sdir, sid, ses):
    csvs = sorted(p for p in sdir.glob('*_img_*.csv') if not p.name.startswith('._'))
    n = 0
    for tidx, csv in enumerate(csvs):
        word = csv.stem.split('_img_')[0]
        if word not in word2label: continue
        label = word2label[word]
        outs = {m: out_path(m, sid, ses, tidx) for m in METRICS}
        if not OVERWRITE and all(o.exists() for o in outs.values()):
            n += 1; continue
        try:
            x = normalize_trial(load_trial(csv))
        except Exception as e:
            log.warning(f'skip {csv.name}: {e}'); continue
        cons = compute_consensus_mask(x, CONSENSUS_METRICS, EDGE_THRESHOLD_PCT, CONSENSUS_K)
        meta = {'subj': sid, 'sess': ses, 'word': word, 'trial': tidx}
        for m in METRICS:
            save_pt(build_hypergraph_dict(x, m, K_HYPEREDGE, label, meta, cons), outs[m])
        n += 1
    return n

sessions = sorted(d for d in CSV_ROOT.iterdir() if d.is_dir() and _PAT.match(d.name))
log.info(f'Sessioni da processare: {len(sessions)}')
tot = 0
for sd in tqdm(sessions, desc='build grafi'):
    m = _PAT.match(sd.name)
    tot += process_session(sd, int(m.group(1)), int(m.group(2)))
log.info(f'Trial processati: {tot}')


## §6 — Verifica

In [ ]:
# Verifica
import random
for metric in METRICS:
    base = DATA_OUT / f'hypergraphs_pruned_{metric}'
    pts = list(base.rglob('trial_*.pt')) if base.exists() else []
    if not pts:
        print(f'  [hypergraphs_pruned_{metric}]  0 file  ⚠️'); continue
    s = torch.load(random.choice(pts), weights_only=False)
    print(f'  [hypergraphs_pruned_{metric}]  {len(pts):>6} file  '
          f"x={tuple(s['x'].shape)}  H={tuple(s['H'].shape)}  adj={tuple(s['adj'].shape)}  y={int(s['y'])}")
n_subj = len({p.parent.name.split('_')[0] for p in (DATA_OUT/f'hypergraphs_pruned_{METRICS[0]}').rglob('trial_*.pt')})
print(f'Soggetti con grafi: {n_subj}')
